# Aircraft Trajectory Matching Pipeline
## With Correct Schema: sensor_df + event_df

**Objective**: Match aircraft trajectories from sensor data to event reports using embedding-based similarity.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
import uuid
from collections import defaultdict
import json

np.random.seed(42)
random.seed(42)

# 1. Mock Data Generation (Schema-Correct)

In [2]:
N_TRACKS = 500
POINTS_PER_TRACK = 20
N_REPORTS = 200

In [3]:
# Generate sensor data with CORRECT schema
sensor_rows = []

for trackId in range(N_TRACKS):
    lat = np.random.uniform(10, 25)
    lon = np.random.uniform(100, 120)
    heading = np.random.uniform(0, 360)
    speed = np.random.uniform(200, 900)
    
    for t in range(POINTS_PER_TRACK):
        # Update position based on heading and speed
        rad = np.deg2rad(heading)
        lat_delta = speed * np.cos(rad) * 0.00001
        lon_delta = speed * np.sin(rad) * 0.00001
        
        lat += lat_delta + np.random.normal(0, 0.001)
        lon += lon_delta + np.random.normal(0, 0.001)
        heading += np.random.normal(0, 2)
        speed += np.random.normal(0, 20)
        speed = np.clip(speed, 100, 1000)
        
        sensor_rows.append({
            "trackId": trackId,
            "timestamp": t,
            "mode3A": random.randint(1000, 9999),
            "heading": heading % 360,
            "speed": speed,
            "longitude": lon,
            "latitude": lat
        })

sensor_df = pd.DataFrame(sensor_rows)
print("Sensor DF columns:", sensor_df.columns.tolist())
print("Sensor DF shape:", sensor_df.shape)
print(sensor_df.head())

Sensor DF columns: ['trackId', 'timestamp', 'mode3A', 'heading', 'speed', 'longitude', 'latitude']
Sensor DF shape: (10000, 7)
   trackId  timestamp  mode3A     heading       speed   longitude   latitude
0        0          0    2824  266.676245  634.409634  119.007901  15.617169
1        0          1    1409  265.749409  625.095038  119.002110  15.616331
2        0          2    5506  262.299574  613.849288  118.993963  15.616110
3        0          3    5012  260.483525  585.603214  118.988194  15.614275
4        0          4    4657  260.618582  557.108250  118.982193  15.614772


In [4]:
# Generate event/report data with CORRECT schema
event_rows = []
gt_dict = {}  # ground truth: report_id -> track_id

track_groups = sensor_df.groupby("trackId")

for report_id in range(N_REPORTS):
    # Pick random track and segment
    trackId = random.randint(0, N_TRACKS - 1)
    track_data = track_groups.get_group(trackId).sort_values("timestamp")
    
    coords = track_data[["latitude", "longitude"]].values
    
    start_idx = random.randint(0, len(coords) - 5)
    segment_len = random.randint(3, 6)
    segment = coords[start_idx:start_idx + segment_len]
    
    # Add noise
    noise = np.random.normal(0, 0.02, segment.shape)
    segment_noisy = segment + noise
    
    lats = segment_noisy[:, 0].tolist()
    lons = segment_noisy[:, 1].tolist()
    
    event_rows.append({
        "eventMention": f"Aircraft spotted near zone {report_id}",
        "eventType": random.choice(["PATROL", "SURVEILLANCE", "UNKNOWN"]),
        "obj": f"Aircraft_{trackId}",
        "entityId": str(uuid.uuid4()),
        "objType": random.choice(["FIGHTER", "BOMBER", "DRONE"]),
        "nation": random.choice(["US", "CN", "RU", "VN"]),
        "quantity": random.randint(1, 4),
        "locIds": str(uuid.uuid4()),
        "loc": f"Zone_{random.randint(1, 100)}",
        "coors.type": "LineString",
        "coors.properties.basedName": f"Base_{random.randint(1, 10)}",
        "coors.geometry.type": "LineString",
        "coors.longitudes": lons,
        "coors.latitudes": lats
    })
    
    gt_dict[report_id] = trackId

event_df = pd.DataFrame(event_rows)
print("Event DF columns:", event_df.columns.tolist())
print("Event DF shape:", event_df.shape)
print(event_df.head())

Event DF columns: ['eventMention', 'eventType', 'obj', 'entityId', 'objType', 'nation', 'quantity', 'locIds', 'loc', 'coors.type', 'coors.properties.basedName', 'coors.geometry.type', 'coors.longitudes', 'coors.latitudes']
Event DF shape: (200, 14)
                   eventMention     eventType           obj  \
0  Aircraft spotted near zone 0  SURVEILLANCE  Aircraft_478   
1  Aircraft spotted near zone 1       UNKNOWN  Aircraft_230   
2  Aircraft spotted near zone 2        PATROL  Aircraft_215   
3  Aircraft spotted near zone 3  SURVEILLANCE  Aircraft_489   
4  Aircraft spotted near zone 4  SURVEILLANCE  Aircraft_358   

                               entityId  objType nation  quantity  \
0  d491ed8a-30d0-45dc-8843-06a96f2c6c1a  FIGHTER     VN         3   
1  6e2d40f5-1c57-4522-8a5e-0b4d981de43c    DRONE     US         1   
2  5dd1c8da-bb1a-4b36-bde0-b9430eed67d5   BOMBER     CN         2   
3  22a71c6a-c648-4557-97a1-465f2b17c4eb  FIGHTER     RU         3   
4  9d250ade-2a75-467f-895f-

In [5]:
# Save to parquet
sensor_df.to_parquet("sensor.parquet", index=False)
event_df.to_parquet("event.parquet", index=False)

# Save ground truth
gt_df = pd.DataFrame([
    {"report_id": rid, "track_id": tid}
    for rid, tid in gt_dict.items()
])
gt_df.to_parquet("ground_truth.parquet", index=False)

print("âœ… Saved: sensor.parquet, event.parquet, ground_truth.parquet")

âœ… Saved: sensor.parquet, event.parquet, ground_truth.parquet


# 2. Pipeline Functions

In [6]:
def build_trajectories(sensor_df):
    """Extract trajectories from sensor data grouped by trackId"""
    sensor_df = sensor_df.sort_values(["trackId", "timestamp"])
    
    trajs = {}
    for trackId, group in sensor_df.groupby("trackId"):
        # Extract: [latitude, longitude, heading, speed, timestamp]
        trajs[trackId] = group[["latitude", "longitude", "heading", "speed", "timestamp"]].values
    
    return trajs

In [7]:
def split_sub_trajectories(trajs, min_len=4, max_len=8, stride=2):
    """Split each trajectory into sub-segments"""
    subs = []
    
    for track_id, traj in trajs.items():
        n = len(traj)
        
        for w in range(min_len, max_len + 1):
            for i in range(0, n - w + 1, stride):
                seg = traj[i:i + w]
                subs.append({
                    "track_id": track_id,
                    "segment": seg,
                    "t_start": seg[0, -1],
                    "t_end": seg[-1, -1]
                })
    
    return subs

In [8]:
def encode_subtrack(seg):
    """Encode sub-trajectory segment to feature vector
    seg: [latitude, longitude, heading, speed, timestamp] shape (N, 5)
    """
    xy = seg[:, :2]  # lat, lon
    heading = seg[:, 2]
    speed = seg[:, 3]
    
    start = xy[0]
    end = xy[-1]
    
    # Direction vector
    vec = end - start
    length = np.linalg.norm(vec) + 1e-6
    direction = vec / length
    
    # Curvature from path
    diffs = np.diff(xy, axis=0)
    if len(diffs) > 0:
        angles = np.arctan2(diffs[:, 1], diffs[:, 0])
        curvature = np.std(angles)
    else:
        curvature = 0.0
    
    speed_mean = np.mean(speed)
    speed_std = np.std(speed)
    
    # Heading stats
    heading_rad = np.deg2rad(heading)
    heading_vec = [
        np.cos(np.mean(heading_rad)),
        np.sin(np.mean(heading_rad))
    ]
    
    return np.array([
        start[0], start[1],          # start position
        end[0], end[1],              # end position
        direction[0], direction[1],  # direction
        length,                      # distance
        curvature,                   # path curvature
        speed_mean, speed_std,       # speed stats
        heading_vec[0], heading_vec[1]  # heading vector
    ], dtype=np.float32)

In [9]:
def safe_array(x):
    """Safely convert any input to float array"""
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    if isinstance(x, (list, tuple)):
        return np.array(x, dtype=np.float32)
    if pd.isna(x):
        return np.zeros(0, dtype=np.float32)
    if isinstance(x, str):
        try:
            parsed = eval(x)
            if isinstance(parsed, (list, tuple, np.ndarray)):
                return np.array(parsed, dtype=np.float32)
        except Exception:
            pass
    try:
        return np.array(x, dtype=np.float32)
    except Exception:
        return np.zeros(0, dtype=np.float32)

In [10]:
def encode_report(row, k=5):
    """Encode event report to feature vector by sampling coordinates"""
    lons = safe_array(row.get("coors.longitudes", []))
    lats = safe_array(row.get("coors.latitudes", []))
    
    if len(lons) == 0 or len(lats) == 0:
        return np.zeros(2 * k + 1, dtype=np.float32)
    
    if len(lons) != len(lats):
        n = min(len(lons), len(lats))
        lons = lons[:n]
        lats = lats[:n]
    
    # Pad if too short
    if len(lons) < k:
        lons = np.pad(lons, (0, k - len(lons)), constant_values=0.0)
        lats = np.pad(lats, (0, k - len(lats)), constant_values=0.0)
    
    # Sample k points uniformly
    idx = np.linspace(0, len(lons) - 1, k).astype(int)
    pts = np.stack([lats[idx], lons[idx]], axis=1)
    
    # Curvature
    if pts.shape[0] < 2:
        curvature = 0.0
    else:
        diffs = np.diff(pts, axis=0)
        curvature = np.std(np.arctan2(diffs[:, 1], diffs[:, 0]))
    
    return np.concatenate([pts.flatten(), [curvature]]).astype(np.float32)

In [37]:
def encode_subtrack_enhanced(seg):
    """Enhanced sub-trajectory encoding with temporal features"""
    xy = seg[:, :2]  # lat, lon
    heading = seg[:, 2]
    speed = seg[:, 3]
    timestamps = seg[:, 4]

    start = xy[0]
    end = xy[-1]

    # Direction vector
    vec = end - start
    length = np.linalg.norm(vec) + 1e-6
    direction = vec / length

    # Curvature from path
    diffs = np.diff(xy, axis=0)
    if len(diffs) > 0:
        angles = np.arctan2(diffs[:, 1], diffs[:, 0])
        curvature = np.std(angles)
    else:
        curvature = 0.0

    # Speed statistics
    speed_mean = np.mean(speed)
    speed_std = np.std(speed)
    speed_min = np.min(speed)
    speed_max = np.max(speed)

    # Heading statistics
    heading_rad = np.deg2rad(heading)
    heading_mean = np.mean(heading_rad)
    heading_std = np.std(heading_rad)

    # Heading vector (circular mean)
    heading_vec_cos = np.cos(heading_mean)
    heading_vec_sin = np.sin(heading_mean)

    # Temporal features
    time_span = timestamps[-1] - timestamps[0]
    time_span = max(time_span, 1.0)  # Avoid division by zero

    # Speed change rate
    if len(speed) > 1:
        speed_changes = np.diff(speed)
        accel_mean = np.mean(speed_changes)
        accel_std = np.std(speed_changes)
    else:
        accel_mean = 0.0
        accel_std = 0.0

    # Heading change rate
    if len(heading) > 1:
        heading_changes = np.diff(heading_rad)
        turn_rate_mean = np.mean(heading_changes)
        turn_rate_std = np.std(heading_changes)
    else:
        turn_rate_mean = 0.0
        turn_rate_std = 0.0

    return np.array([
        # Spatial features (6)
        start[0], start[1],          # start position
        end[0], end[1],              # end position
        direction[0], direction[1],  # direction

        # Geometric features (2)
        length,                      # distance
        curvature,                   # path curvature

        # Speed features (4)
        speed_mean, speed_std, speed_min, speed_max,

        # Heading features (4)
        heading_vec_cos, heading_vec_sin,
        heading_std,                 # heading variability

        # Temporal features (5)
        time_span,                   # total duration
        accel_mean, accel_std,       # acceleration stats
        turn_rate_mean, turn_rate_std # turning rate stats
    ], dtype=np.float32)

print("âœ… Enhanced subtrack encoding with temporal features defined")

âœ… Enhanced subtrack encoding with temporal features defined


In [11]:
def build_features(sub_tracks, event_df):
    """Build normalized feature vectors for both sub-tracks and reports"""
    sub_vecs = np.stack([encode_subtrack(s["segment"]) for s in sub_tracks])
    report_vecs = np.stack([encode_report(row) for _, row in event_df.iterrows()])
    
    # Normalize
    sub_vecs = (sub_vecs - sub_vecs.mean(0)) / (sub_vecs.std(0) + 1e-6)
    report_vecs = (report_vecs - report_vecs.mean(0)) / (report_vecs.std(0) + 1e-6)
    
    return sub_vecs, report_vecs

In [12]:
def filter_candidates(sub_vecs, report_vecs, radius=0.05, max_k=10):
    """Filter nearby report candidates for each sub-track"""
    candidates = []
    
    for s in sub_vecs:
        # Distance to first 2 features (start position)
        d = np.linalg.norm(report_vecs[:, :2] - s[:2], axis=1)
        idx = np.where(d < radius)[0]
        
        if len(idx) == 0:
            idx = np.argsort(d)[:max_k]
        
        candidates.append(idx)
    
    return candidates

In [13]:
def create_unsupervised_gt(sensor_df, event_df, spatial_thresh=0.05, temporal_thresh=10):
    """
    Create pseudo ground truth labels in unsupervised manner using spatiotemporal proximity
    
    Args:
        sensor_df: Sensor data with columns [trackId, timestamp, latitude, longitude, ...]
        event_df: Event data with columns [..., coors.latitudes, coors.longitudes, ...]
        spatial_thresh: Maximum lat/lon distance (degrees) to consider a match
        temporal_thresh: Maximum timestamp difference to consider temporal proximity
    
    Returns:
        gt_dict: {report_id -> track_id} pseudo ground truth mapping
        confidence_scores: {report_id -> confidence_score} confidence of each match (0-1)
    """
    gt_dict = {}
    confidence_scores = {}
    
    # Group sensor data by trackId for faster lookup
    track_groups = sensor_df.groupby("trackId")
    
    for report_id, row in event_df.iterrows():
        # Extract report coordinates
        lons = safe_array(row.get("coors.longitudes", []))
        lats = safe_array(row.get("coors.latitudes", []))
        
        if len(lons) == 0 or len(lats) == 0:
            continue
        
        # Calculate report centroid
        report_lat = np.mean(lats)
        report_lon = np.mean(lons)
        report_bbox = {
            "lat_min": np.min(lats) - spatial_thresh,
            "lat_max": np.max(lats) + spatial_thresh,
            "lon_min": np.min(lons) - spatial_thresh,
            "lon_max": np.max(lons) + spatial_thresh
        }
        
        best_track_id = None
        best_score = 0.0
        
        # Search all tracks
        for track_id, track_group in track_groups:
            track_df = track_group.sort_values("timestamp")
            
            # Extract track positions
            track_lats = track_df["latitude"].values
            track_lons = track_df["longitude"].values
            track_times = track_df["timestamp"].values
            
            # Count points inside report bounding box
            inside_bbox = (
                (track_lats >= report_bbox["lat_min"]) & 
                (track_lats <= report_bbox["lat_max"]) &
                (track_lons >= report_bbox["lon_min"]) & 
                (track_lons <= report_bbox["lon_max"])
            )
            
            num_inside = np.sum(inside_bbox)
            
            if num_inside == 0:
                continue
            
            # Calculate spatial score: ratio of points within bbox
            spatial_score = num_inside / len(track_lats)
            
            # Calculate overlap with report time range (if available)
            temporal_score = 1.0
            if "timestamp" in track_df.columns and len(track_times) > 0:
                track_time_range = track_times[-1] - track_times[0]
                if track_time_range > 0:
                    temporal_score = min(1.0, track_time_range / temporal_thresh)
            
            # Combined score: weighted average
            combined_score = 0.7 * spatial_score + 0.3 * temporal_score
            
            # Track the best match
            if combined_score > best_score:
                best_score = combined_score
                best_track_id = track_id
        
        # Add to ground truth if confidence above 0.3
        if best_track_id is not None and best_score >= 0.3:
            gt_dict[report_id] = best_track_id
            confidence_scores[report_id] = best_score
    
    return gt_dict, confidence_scores

print("âœ… Unsupervised GT creation function defined")

âœ… Unsupervised GT creation function defined


In [14]:
def create_pairs(sub_vecs, report_vecs, candidates, k_pos=2, k_neg=4):
    """Create positive/negative pairs for training"""
    pairs, labels = [], []
    
    for i, cand in enumerate(candidates):
        d = np.linalg.norm(report_vecs[cand, :2] - sub_vecs[i][:2], axis=1)
        order = cand[np.argsort(d)]
        
        pos = order[:k_pos]
        neg = order[k_pos:k_pos + k_neg]
        
        for p in pos:
            pairs.append((i, p))
            labels.append(1)
        
        for n in neg:
            pairs.append((i, n))
            labels.append(0)
    
    return np.array(pairs), np.array(labels)

In [15]:
def create_triplets_gt(sub_tracks, gt_dict, num_pos=3, num_neg=8):
    """Create triplets from ground truth labels"""
    track_to_sub = defaultdict(list)
    for idx, s in enumerate(sub_tracks):
        track_to_sub[s["track_id"]].append(idx)
    
    all_indices = np.arange(len(sub_tracks))
    triplets = []
    
    for rid, true_tid in gt_dict.items():
        pos_candidates = track_to_sub.get(true_tid, [])
        if len(pos_candidates) == 0:
            continue
        
        pos_sample = random.sample(pos_candidates, min(num_pos, len(pos_candidates)))
        
        neg_candidates = all_indices[[
            sub_tracks[i]["track_id"] != true_tid for i in all_indices
        ]]
        
        if len(neg_candidates) == 0:
            continue
        
        neg_sample = random.sample(list(neg_candidates), min(num_neg, len(neg_candidates)))
        
        for p in pos_sample:
            for n in neg_sample:
                triplets.append((rid, p, n))
    
    return triplets

In [41]:
def create_triplets_hard_negative(sub_tracks, gt_dict, sub_vecs, report_vecs, net_t, net_r,
                                  num_pos=3, num_neg=8, hard_neg_ratio=0.5):
    """
    Create triplets with hard negative mining using current model embeddings
    """
    track_to_sub = defaultdict(list)
    for idx, s in enumerate(sub_tracks):
        track_to_sub[s["track_id"]].append(idx)

    all_indices = np.arange(len(sub_tracks))
    triplets = []

    # Get current embeddings for hard negative mining
    with torch.no_grad():
        e_t = net_t(torch.tensor(sub_vecs))
        e_r = net_r(torch.tensor(report_vecs))

    for rid, true_tid in gt_dict.items():
        pos_candidates = track_to_sub.get(true_tid, [])
        if len(pos_candidates) == 0:
            continue

        pos_sample = random.sample(pos_candidates, min(num_pos, len(pos_candidates)))

        # Hard negative mining: select negatives that are close to anchor in embedding space
        anchor_emb = e_r[rid]  # report embedding
        all_neg_emb = e_t  # all subtrack embeddings

        # Compute distances to anchor
        distances = torch.cdist(anchor_emb.unsqueeze(0), all_neg_emb).squeeze(0)
        distances = distances.cpu().numpy()

        # Get hard negatives (closest wrong tracks)
        neg_candidates = []
        for idx in all_indices:
            if sub_tracks[idx]["track_id"] != true_tid:
                neg_candidates.append((idx, distances[idx]))

        # Sort by distance (closest first = hardest)
        neg_candidates.sort(key=lambda x: x[1])

        # Sample mix of hard and random negatives
        num_hard = int(num_neg * hard_neg_ratio)
        num_random = num_neg - num_hard

        hard_negs = [idx for idx, _ in neg_candidates[:num_hard]]

        # Random negatives from remaining
        remaining_negs = [idx for idx, _ in neg_candidates[num_hard:]]
        if len(remaining_negs) >= num_random:
            random_negs = random.sample(remaining_negs, num_random)
        else:
            random_negs = remaining_negs

        neg_sample = hard_negs + random_negs

        # Ensure we have enough negatives
        if len(neg_sample) < num_neg:
            # Fill with random if needed
            all_possible_negs = [idx for idx in all_indices if sub_tracks[idx]["track_id"] != true_tid]
            additional_negs = random.sample(all_possible_negs, min(num_neg - len(neg_sample), len(all_possible_negs)))
            neg_sample.extend(additional_negs)

        neg_sample = neg_sample[:num_neg]  # Trim if too many

        for p in pos_sample:
            for n in neg_sample:
                triplets.append((rid, p, n))

    return triplets

class TripletLossHard(nn.Module):
    """Triplet loss with hard negative mining and adaptive margin"""
    def __init__(self, margin=1.0, hard_margin=0.2):
        super().__init__()
        self.margin = margin
        self.hard_margin = hard_margin

    def forward(self, anchor, pos, neg):
        pos_dist = F.pairwise_distance(anchor, pos)
        neg_dist = F.pairwise_distance(anchor, neg)

        # Standard triplet loss
        basic_loss = F.relu(pos_dist - neg_dist + self.margin)

        # Hard negative loss (penalize more when negative is very close)
        hard_loss = F.relu(pos_dist - neg_dist + self.hard_margin)

        # Combine losses and return scalar
        combined_loss = 0.7 * basic_loss + 0.3 * hard_loss
        return combined_loss.mean()  # Ensure scalar output

print("âœ… Hard negative mining and enhanced triplet loss defined")

âœ… Hard negative mining and enhanced triplet loss defined


# 3. Neural Network Models

In [16]:
class EmbeddingNet(nn.Module):
    def __init__(self, in_dim, emb_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, emb_dim)
        )
    
    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        return (y * d**2 + (1 - y) * F.relu(self.margin - d)**2).mean()

class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    
    def forward(self, anchor, pos, neg):
        pos_dist = F.pairwise_distance(anchor, pos)
        neg_dist = F.pairwise_distance(anchor, neg)
        return F.relu(pos_dist - neg_dist + self.margin).mean()

In [17]:
class SubTrackDataset(Dataset):
    def __init__(self, pairs, labels, sub_vecs, report_vecs):
        self.pairs = pairs
        self.labels = labels
        self.sub_vecs = sub_vecs
        self.report_vecs = report_vecs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        i, j = self.pairs[idx]
        return (
            torch.tensor(self.sub_vecs[i]),
            torch.tensor(self.report_vecs[j]),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

class ReportTripletDataset(Dataset):
    def __init__(self, triplets, sub_vecs, report_vecs):
        self.triplets = triplets
        self.sub_vecs = sub_vecs
        self.report_vecs = report_vecs
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        rid, pos_idx, neg_idx = self.triplets[idx]
        return (
            torch.tensor(self.report_vecs[rid]),
            torch.tensor(self.sub_vecs[pos_idx]),
            torch.tensor(self.sub_vecs[neg_idx])
        )

In [18]:
def train_model_contrastive(sub_vecs, report_vecs, pairs, labels, epochs=10, emb_dim=64):
    """Train embedding networks with contrastive loss"""
    dataset = SubTrackDataset(pairs, labels, sub_vecs, report_vecs)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)
    
    net_t = EmbeddingNet(sub_vecs.shape[1], emb_dim)
    net_r = EmbeddingNet(report_vecs.shape[1], emb_dim)
    
    optimizer = torch.optim.Adam(
        list(net_t.parameters()) + list(net_r.parameters()),
        lr=1e-3
    )
    loss_fn = ContrastiveLoss()
    
    for ep in range(epochs):
        total_loss = 0
        for t_vec, r_vec, label in loader:
            optimizer.zero_grad()
            e_t = net_t(t_vec)
            e_r = net_r(r_vec)
            loss = loss_fn(e_t, e_r, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {ep + 1}: loss={total_loss / len(loader):.4f}")
    
    return net_t, net_r

In [19]:
def train_model_triplet(sub_vecs, report_vecs, triplets, epochs=500, emb_dim=128):
    """Train embedding networks with triplet loss using GT labels"""
    dataset = ReportTripletDataset(triplets, sub_vecs, report_vecs)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)
    
    net_r = EmbeddingNet(report_vecs.shape[1], emb_dim)
    net_t = EmbeddingNet(sub_vecs.shape[1], emb_dim)
    
    optimizer = torch.optim.Adam(
        list(net_r.parameters()) + list(net_t.parameters()),
        lr=3e-4
    )
    loss_fn = TripletLoss(margin=1.0)
    
    for ep in range(epochs):
        total_loss = 0.0
        for r_vec, pos_vec, neg_vec in loader:
            optimizer.zero_grad()
            e_r = net_r(r_vec)
            e_pos = net_t(pos_vec)
            e_neg = net_t(neg_vec)
            loss = loss_fn(e_r, e_pos, e_neg)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (ep + 1) % 50 == 0 or ep == 0:
            print(f"Epoch {ep + 1}: loss={total_loss / len(loader):.4f}")
    
    return net_t, net_r

In [39]:
def train_model_triplet_enhanced(sub_vecs, report_vecs, triplets, epochs=100, emb_dim=256,
                                 hard_negative_epochs=20):
    """
    Enhanced triplet training with hard negative mining and larger embeddings
    """
    dataset = ReportTripletDataset(triplets, sub_vecs, report_vecs)
    loader = DataLoader(dataset, batch_size=128, shuffle=True)

    net_r = EmbeddingNet(report_vecs.shape[1], emb_dim)
    net_t = EmbeddingNet(sub_vecs.shape[1], emb_dim)

    optimizer = torch.optim.Adam(
        list(net_r.parameters()) + list(net_t.parameters()),
        lr=3e-4
    )

    # Use standard triplet loss initially, then switch to hard negative loss
    loss_fn_standard = TripletLoss(margin=1.5)  # Increased margin
    loss_fn_hard = TripletLossHard(margin=1.5, hard_margin=0.3)

    for ep in range(epochs):
        if ep < hard_negative_epochs:
            loss_fn = loss_fn_standard
        else:
            loss_fn = loss_fn_hard

        total_loss = 0.0
        for r_vec, pos_vec, neg_vec in loader:
            optimizer.zero_grad()
            e_r = net_r(r_vec)
            e_pos = net_t(pos_vec)
            e_neg = net_t(neg_vec)
            loss = loss_fn(e_r, e_pos, e_neg)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (ep + 1) % 20 == 0 or ep == 0:
            loss_type = "standard" if ep < hard_negative_epochs else "hard-neg"
            print(f"Epoch {ep + 1}: loss={total_loss / len(loader):.4f} ({loss_type})")

    return net_t, net_r

def build_features_enhanced(sub_tracks, event_df):
    """Build enhanced features with temporal information"""
    sub_vecs = np.stack([encode_subtrack_enhanced(s["segment"]) for s in sub_tracks])
    report_vecs = np.stack([encode_report(row) for _, row in event_df.iterrows()])

    # Normalize per feature type for better training
    # Spatial features (0-5): lat/lon coordinates
    sub_vecs[:, :6] = (sub_vecs[:, :6] - sub_vecs[:, :6].mean(0)) / (sub_vecs[:, :6].std(0) + 1e-6)

    # Geometric features (6-7): distance, curvature
    sub_vecs[:, 6:8] = (sub_vecs[:, 6:8] - sub_vecs[:, 6:8].mean(0)) / (sub_vecs[:, 6:8].std(0) + 1e-6)

    # Speed features (8-11): speed statistics
    sub_vecs[:, 8:12] = (sub_vecs[:, 8:12] - sub_vecs[:, 8:12].mean(0)) / (sub_vecs[:, 8:12].std(0) + 1e-6)

    # Heading features (12-15): heading vectors and std
    sub_vecs[:, 12:16] = (sub_vecs[:, 12:16] - sub_vecs[:, 12:16].mean(0)) / (sub_vecs[:, 12:16].std(0) + 1e-6)

    # Temporal features (16-20): time, acceleration, turning
    sub_vecs[:, 16:] = (sub_vecs[:, 16:] - sub_vecs[:, 16:].mean(0)) / (sub_vecs[:, 16:].std(0) + 1e-6)

    # Report vectors (unchanged)
    report_vecs = (report_vecs - report_vecs.mean(0)) / (report_vecs.std(0) + 1e-6)

    return sub_vecs, report_vecs

print("âœ… Enhanced training and feature building functions defined")

âœ… Enhanced training and feature building functions defined


# 4. Matching & Evaluation

In [20]:
def match_tracks(net_t, net_r, sub_tracks, sub_vecs, report_vecs, thresh=0.3):
    """Find best report match for each track"""
    with torch.no_grad():
        e_t = net_t(torch.tensor(sub_vecs))
        e_r = net_r(torch.tensor(report_vecs))
        dist = torch.cdist(e_t, e_r)
    
    track_best = {}
    
    for i, s in enumerate(sub_tracks):
        tid = s["track_id"]
        
        best_report = dist[i].argmin().item()
        best_score = dist[i].min().item()
        
        if best_score < thresh:
            if tid not in track_best or best_score < track_best[tid]["score"]:
                track_best[tid] = {
                    "report_id": best_report,
                    "score": best_score,
                    "sub_idx": i
                }
    
    return track_best

In [21]:
def match_reports_topk(net_t, net_r, sub_tracks, sub_vecs, report_vecs, top_k=5):
    """Find top-K track matches for each report"""
    with torch.no_grad():
        e_t = net_t(torch.tensor(sub_vecs))
        e_r = net_r(torch.tensor(report_vecs))
        dist = torch.cdist(e_r, e_t)  # [num_reports, num_subtracks]
    
    report_topk = {}
    for rid in range(dist.shape[0]):
        row = dist[rid].cpu().numpy()
        order = np.argsort(row)[:top_k]
        topk = []
        for idx in order:
            topk.append({
                "track_id": int(sub_tracks[idx]["track_id"]),
                "sub_idx": int(idx),
                "score": float(row[idx])
            })
        report_topk[rid] = topk
    return report_topk

In [22]:
def evaluate_accuracy(results, gt_dict):
    """Evaluate track-to-report accuracy"""
    correct = 0
    total = 0
    for tid, v in results.items():
        rid = v["report_id"]
        true_tid = gt_dict.get(rid)
        if true_tid is not None:
            total += 1
            if tid == true_tid:
                correct += 1
    accuracy = correct / total if total > 0 else 0
    print(f"Track Accuracy: {accuracy:.2%} ({correct}/{total})")
    return accuracy

def evaluate_topk(report_topk, gt_dict, k=5):
    """Evaluate report-to-track recall@k"""
    hits = 0
    total = 0
    for rid, topk_list in report_topk.items():
        true_tid = gt_dict.get(rid)
        if true_tid is None:
            continue
        total += 1
        top_tracks = [item["track_id"] for item in topk_list[:k]]
        if true_tid in top_tracks:
            hits += 1
    recall = hits / total if total > 0 else 0
    print(f"Report Recall@{k}: {recall:.2%} ({hits}/{total})")
    return recall

# 5. Main Pipeline Execution

In [23]:
# Load data
sensor_df = pd.read_parquet("sensor.parquet")
event_df = pd.read_parquet("event.parquet")
gt_df = pd.read_parquet("ground_truth.parquet")

# Build GT dict
gt_dict = dict(zip(gt_df["report_id"], gt_df["track_id"]))

print(f"Loaded: {len(sensor_df)} sensor points, {len(event_df)} reports, {len(gt_dict)} GT pairs")

Loaded: 10000 sensor points, 200 reports, 200 GT pairs


In [24]:
# Option 1: Use pre-existing ground truth (if available)
if False:  # Set to True if you have ground_truth.parquet
    gt_df = pd.read_parquet("ground_truth.parquet")
    gt_dict = dict(zip(gt_df["report_id"], gt_df["track_id"]))
    print(f"Loaded {len(gt_dict)} ground truth pairs from file")

# Option 2: Generate unsupervised pseudo ground truth (for real data without labels)
else:
    gt_dict, confidence_scores = create_unsupervised_gt(
        sensor_df, 
        event_df, 
        spatial_thresh=0.05, 
        temporal_thresh=10
    )
    print(f"\nðŸ” Generated {len(gt_dict)} unsupervised GT pairs")
    
    if len(confidence_scores) > 0:
        conf_scores = list(confidence_scores.values())
        print(f"Confidence scores: mean={np.mean(conf_scores):.3f}, "
              f"min={np.min(conf_scores):.3f}, max={np.max(conf_scores):.3f}")
        
        # Visualize confidence distribution
        print(f"\nConfidence distribution:")
        print(f"  [0.3-0.4): {sum(1 for s in conf_scores if 0.3 <= s < 0.4)} pairs")
        print(f"  [0.4-0.6): {sum(1 for s in conf_scores if 0.4 <= s < 0.6)} pairs")
        print(f"  [0.6-0.8): {sum(1 for s in conf_scores if 0.6 <= s < 0.8)} pairs")
        print(f"  [0.8-1.0]: {sum(1 for s in conf_scores if 0.8 <= s <= 1.0)} pairs")


ðŸ” Generated 200 unsupervised GT pairs
Confidence scores: mean=0.939, min=0.650, max=1.000

Confidence distribution:
  [0.3-0.4): 0 pairs
  [0.4-0.6): 0 pairs
  [0.6-0.8): 26 pairs
  [0.8-1.0]: 174 pairs


In [28]:
# Pipeline: Basic contrastive training
print("\n=== Pipeline 1: Contrastive Learning ===")

trajs = build_trajectories(sensor_df)
print(f"Built {len(trajs)} trajectories")

sub_tracks = split_sub_trajectories(trajs)
print(f"Split into {len(sub_tracks)} sub-tracks")

sub_vecs, report_vecs = build_features(sub_tracks, event_df)
print(f"Features: sub_tracks {sub_vecs.shape}, reports {report_vecs.shape}")

candidates = filter_candidates(sub_vecs, report_vecs)
pairs, labels = create_pairs(sub_vecs, report_vecs, candidates)
print(f"Created {len(pairs)} pairs")

net_t, net_r = train_model_contrastive(sub_vecs, report_vecs, pairs, labels, epochs=5)

results = match_tracks(net_t, net_r, sub_tracks, sub_vecs, report_vecs)
evaluate_accuracy(results, gt_dict)

# Store for visualization
match_results_contrastive = results

# Save results
df_results = pd.DataFrame([
    {
        "track_id": tid,
        "report_id": v["report_id"],
        "score": v["score"],
        "sub_idx": v["sub_idx"]
    }
    for tid, v in results.items()
])
df_results.to_csv("match_results_contrastive.csv", index=False)
print("âœ… Saved: match_results_contrastive.csv")


=== Pipeline 1: Contrastive Learning ===


Built 500 trajectories
Split into 19500 sub-tracks
Features: sub_tracks (19500, 12), reports (200, 11)
Created 98609 pairs
Epoch 1: loss=0.2240
Epoch 2: loss=0.2089
Epoch 3: loss=0.1999
Epoch 4: loss=0.1925
Epoch 5: loss=0.1850
Track Accuracy: 8.70% (10/115)
âœ… Saved: match_results_contrastive.csv


In [ ]:
import numpy as np
import plotly.graph_objects as go


def _extract_report_coords(row):
    lons = safe_array(row.get("coors.longitudes", []))
    lats = safe_array(row.get("coors.latitudes", []))
    n = min(len(lons), len(lats))
    if n == 0:
        return np.zeros((0, 2), dtype=np.float32)
    return np.column_stack([lats[:n], lons[:n]]).astype(np.float32)


def _add_xy_trace(fig, coords, name, color, width=3, mode='lines', dash=None, opacity=0.9):
    if len(coords) == 0:
        return
    fig.add_trace(go.Scatter(
        x=coords[:, 1],
        y=coords[:, 0],
        mode=mode,
        name=name,
        line=dict(color=color, width=width, dash=dash),
        marker=dict(size=7, color=color),
        opacity=opacity,
    ))


def _finalize_xy_layout(fig, title):
    fig.update_layout(
        title=title,
        template='plotly_white',
        height=700,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        margin=dict(l=40, r=20, t=80, b=40),
        xaxis_title='Longitude / X',
        yaxis_title='Latitude / Y',
    )
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    return fig


def plot_matches_contrastive(sensor_df, event_df, sub_tracks, results, num_samples=10):
    """Offline visualization without map tiles: track, report, matched segment on XY axes."""
    fig = go.Figure()
    items = list(results.items())[:num_samples]

    for tid, v in items:
        rid = v['report_id']
        score = v['score']

        track_df = sensor_df[sensor_df['trackId'] == tid].sort_values('timestamp')
        track_coords = track_df[['latitude', 'longitude']].to_numpy(dtype=np.float32)
        _add_xy_trace(fig, track_coords, f'Track {tid} (score={score:.3f})', 'royalblue', width=3, opacity=0.75)

        report_coords = _extract_report_coords(event_df.iloc[rid])
        _add_xy_trace(fig, report_coords, f'Report {rid}', 'firebrick', width=4, mode='lines+markers', opacity=0.95)

        sub_idx = v['sub_idx']
        if sub_idx < len(sub_tracks):
            seg = sub_tracks[sub_idx]['segment'][:, :2].astype(np.float32)
            _add_xy_trace(fig, seg, f'Matched segment (track {tid})', 'seagreen', width=5, opacity=1.0)

    return _finalize_xy_layout(fig, 'Contrastive Matching Results (Offline XY View)')


def plot_report_topk_matches(sensor_df, event_df, sub_tracks, report_topk, top_k=5, num_reports=10):
    """Offline visualization without map tiles for report top-K matching."""
    fig = go.Figure()
    colors = ['royalblue', 'seagreen', 'darkorange', 'purple', 'teal', 'brown', 'gray']
    report_ids = list(report_topk.keys())[:num_reports]

    for rid in report_ids:
        report_coords = _extract_report_coords(event_df.iloc[rid])
        _add_xy_trace(fig, report_coords, f'Report {rid}', 'firebrick', width=4, mode='lines+markers', opacity=0.95)

        candidates = report_topk[rid][:top_k]
        for rank, item in enumerate(candidates):
            tid = item['track_id']
            score = item['score']
            track_df = sensor_df[sensor_df['trackId'] == tid].sort_values('timestamp')
            track_coords = track_df[['latitude', 'longitude']].to_numpy(dtype=np.float32)
            _add_xy_trace(
                fig,
                track_coords,
                f'Report {rid} rank#{rank+1}: Track {tid} ({score:.3f})',
                colors[rank % len(colors)],
                width=2.5,
                opacity=0.65,
            )

    return _finalize_xy_layout(fig, 'Top-K Matching Results (Offline XY View)')

print('Defined offline XY visualization functions')


In [ ]:
# Display contrastive matching results (offline XY)
m_contrastive = plot_matches_contrastive(sensor_df, event_df, sub_tracks, match_results_contrastive, num_samples=10)
m_contrastive.write_html("matches_contrastive.html", include_plotlyjs=True, full_html=True)
print("Saved: matches_contrastive.html (offline self-contained)")
m_contrastive


In [ ]:
# Display triplet (topK) matching results (offline XY)
m_topk = plot_report_topk_matches(sensor_df, event_df, sub_tracks, report_topk_results, top_k=5, num_reports=100)
m_topk.write_html("matches_topk.html", include_plotlyjs=True, full_html=True)
print("Saved: matches_topk.html (offline self-contained)")
m_topk


In [ ]:
import plotly.graph_objects as go


def plot_matches_contrastive_px(sensor_df, event_df, sub_tracks, results, num_samples=10):
    """Alias of offline XY contrastive visualization for backward compatibility."""
    return plot_matches_contrastive(sensor_df, event_df, sub_tracks, results, num_samples=num_samples)


def plot_report_topk_matches_px(sensor_df, event_df, sub_tracks, report_topk, top_k=5, num_reports=10):
    """Alias of offline XY top-K visualization for backward compatibility."""
    return plot_report_topk_matches(sensor_df, event_df, sub_tracks, report_topk, top_k=top_k, num_reports=num_reports)

print('Defined Plotly offline aliases')


In [ ]:
# Display contrastive matching results with Plotly (offline XY)
fig_contrastive = plot_matches_contrastive_px(sensor_df, event_df, sub_tracks, match_results_contrastive, num_samples=10)
fig_contrastive.show()
print("Displayed offline Plotly contrastive visualization")


In [ ]:
# Display triplet (topK) matching results with Plotly (offline XY)
fig_topk = plot_report_topk_matches_px(sensor_df, event_df, sub_tracks, report_topk_results, top_k=5, num_reports=10)
fig_topk.show()
print("Displayed offline Plotly topK visualization")


In [ ]:
# Display enhanced triplet (topK) matching results with Plotly
if 'report_topk_results_enhanced' in globals():
    fig_enhanced = plot_report_topk_matches_px(sensor_df, event_df, sub_tracks,
                                              report_topk_results_enhanced, top_k=5, num_reports=10)
    fig_enhanced.update_layout(title="Enhanced TopK Matching Results (Hard Negatives + Temporal)")
    fig_enhanced.show()
    print("âœ… Plotly enhanced visualization displayed")
else:
    print("âš ï¸  Enhanced results not available")

In [ ]:
# Display optimized triplet (topK) matching results with Plotly
if 'report_topk_results_optimized' in globals():
    fig_optimized = plot_report_topk_matches_px(sensor_df, event_df, sub_tracks,
                                               report_topk_results_optimized, top_k=5, num_reports=10)
    fig_optimized.update_layout(title="Optimized TopK Matching Results (192-dim + Moderate Hard Neg)")
    fig_optimized.show()
    print("âœ… Plotly optimized visualization displayed")
else:
    print("âš ï¸  Optimized results not available")

In [30]:
# Pipeline: Triplet learning with GT labels
print("\n=== Pipeline 2: Triplet Learning (report topK) ===")

triplets = create_triplets_gt(sub_tracks, gt_dict, num_pos=3, num_neg=8)
print(f"Created {len(triplets)} triplets from GT")

if len(triplets) > 0:
    net_t2, net_r2 = train_model_triplet(sub_vecs, report_vecs, triplets, epochs=50, emb_dim=128)
    
    report_topk_results = match_reports_topk(net_t2, net_r2, sub_tracks, sub_vecs, report_vecs, top_k=5)
    evaluate_topk(report_topk_results, gt_dict, k=5)
    
    # Save report->topK results to CSV
    report_topk_csv = []
    for rid, candidates in report_topk_results.items():
        for rank, item in enumerate(candidates[:5], start=1):
            report_topk_csv.append({
                "report_id": rid,
                "rank": rank,
                "track_id": item["track_id"],
                "score": item["score"],
                "sub_idx": item["sub_idx"]
            })
    
    pd.DataFrame(report_topk_csv).to_csv("report_topk_results.csv", index=False)
    print("âœ… Saved: report_topk_results.csv")
else:
    print("âš ï¸  No triplets generated. Check GT dict and sub-tracks.")


=== Pipeline 2: Triplet Learning (report topK) ===
Created 4800 triplets from GT
Epoch 1: loss=0.7880
Epoch 50: loss=0.0704
Report Recall@5: 56.00% (112/200)
âœ… Saved: report_topk_results.csv


In [42]:
# Pipeline: Enhanced triplet learning with hard negatives and temporal features
print("\n=== Pipeline 3: Enhanced Triplet Learning (Improved) ===")

# Build enhanced features
sub_vecs_enhanced, report_vecs_enhanced = build_features_enhanced(sub_tracks, event_df)
print(f"Enhanced features: sub_tracks {sub_vecs_enhanced.shape}, reports {report_vecs_enhanced.shape}")

# Initial triplets for warm-up
triplets_init = create_triplets_gt(sub_tracks, gt_dict, num_pos=3, num_neg=8)
print(f"Created {len(triplets_init)} initial triplets")

if len(triplets_init) > 0:
    # Phase 1: Train with standard triplets
    print("\n--- Phase 1: Standard Triplet Training ---")
    net_t_init, net_r_init = train_model_triplet_enhanced(
        sub_vecs_enhanced, report_vecs_enhanced, triplets_init,
        epochs=40, emb_dim=256, hard_negative_epochs=40  # No hard negatives yet
    )

    # Phase 2: Hard negative mining and fine-tuning
    print("\n--- Phase 2: Hard Negative Mining ---")
    triplets_hard = create_triplets_hard_negative(
        sub_tracks, gt_dict, sub_vecs_enhanced, report_vecs_enhanced,
        net_t_init, net_r_init, num_pos=3, num_neg=8, hard_neg_ratio=0.7
    )
    print(f"Created {len(triplets_hard)} hard negative triplets")

    net_t_enhanced, net_r_enhanced = train_model_triplet_enhanced(
        sub_vecs_enhanced, report_vecs_enhanced, triplets_hard,
        epochs=60, emb_dim=256, hard_negative_epochs=20  # Enable hard negatives
    )

    # Evaluate enhanced model
    report_topk_enhanced = match_reports_topk(net_t_enhanced, net_r_enhanced, sub_tracks,
                                             sub_vecs_enhanced, report_vecs_enhanced, top_k=5)
    evaluate_topk(report_topk_enhanced, gt_dict, k=5)

    # Save enhanced results
    report_topk_enhanced_csv = []
    for rid, candidates in report_topk_enhanced.items():
        for rank, item in enumerate(candidates[:5], start=1):
            report_topk_enhanced_csv.append({
                "report_id": rid,
                "rank": rank,
                "track_id": item["track_id"],
                "score": item["score"],
                "sub_idx": item["sub_idx"]
            })

    pd.DataFrame(report_topk_enhanced_csv).to_csv("report_topk_enhanced.csv", index=False)
    print("âœ… Saved: report_topk_enhanced.csv")

    # Store for visualization
    report_topk_results_enhanced = report_topk_enhanced

else:
    print("âš ï¸  No triplets generated. Check GT dict and sub-tracks.")


=== Pipeline 3: Enhanced Triplet Learning (Improved) ===


Enhanced features: sub_tracks (19500, 20), reports (200, 11)
Created 4800 initial triplets

--- Phase 1: Standard Triplet Training ---
Epoch 1: loss=1.3328 (standard)
Epoch 20: loss=0.3865 (standard)
Epoch 40: loss=0.3306 (standard)

--- Phase 2: Hard Negative Mining ---
Created 4800 hard negative triplets
Epoch 1: loss=1.4296 (standard)
Epoch 20: loss=0.5635 (standard)
Epoch 40: loss=0.3866 (hard-neg)
Epoch 60: loss=0.3322 (hard-neg)
Report Recall@5: 18.00% (36/200)
âœ… Saved: report_topk_enhanced.csv


In [44]:
# Pipeline: Optimized Enhanced Triplet Learning
print("\n=== Pipeline 4: Optimized Enhanced Triplet Learning ===")

# Use original features but with larger embeddings and moderate hard negatives
sub_vecs_optimized, report_vecs_optimized = build_features(sub_tracks, event_df)  # Original features
print(f"Optimized features: sub_tracks {sub_vecs_optimized.shape}, reports {report_vecs_optimized.shape}")

triplets_opt = create_triplets_gt(sub_tracks, gt_dict, num_pos=3, num_neg=8)
print(f"Created {len(triplets_opt)} optimized triplets")

if len(triplets_opt) > 0:
    # Single-phase training with larger embeddings but moderate hard negatives
    net_t_opt, net_r_opt = train_model_triplet_enhanced(
        sub_vecs_optimized, report_vecs_optimized, triplets_opt,
        epochs=80, emb_dim=192, hard_negative_epochs=40  # Moderate hard negatives
    )

    # Evaluate optimized model
    report_topk_optimized = match_reports_topk(net_t_opt, net_r_opt, sub_tracks,
                                              sub_vecs_optimized, report_vecs_optimized, top_k=5)
    evaluate_topk(report_topk_optimized, gt_dict, k=5)

    # Save optimized results
    report_topk_opt_csv = []
    for rid, candidates in report_topk_optimized.items():
        for rank, item in enumerate(candidates[:5], start=1):
            report_topk_opt_csv.append({
                "report_id": rid,
                "rank": rank,
                "track_id": item["track_id"],
                "score": item["score"],
                "sub_idx": item["sub_idx"]
            })

    pd.DataFrame(report_topk_opt_csv).to_csv("report_topk_optimized.csv", index=False)
    print("âœ… Saved: report_topk_optimized.csv")

    # Store for visualization
    report_topk_results_optimized = report_topk_optimized

else:
    print("âš ï¸  No triplets generated.")


=== Pipeline 4: Optimized Enhanced Triplet Learning ===
Optimized features: sub_tracks (19500, 12), reports (200, 11)
Created 4800 optimized triplets
Epoch 1: loss=1.2937 (standard)
Epoch 20: loss=0.4032 (standard)
Epoch 40: loss=0.3026 (standard)
Epoch 60: loss=0.1857 (hard-neg)
Epoch 80: loss=0.1793 (hard-neg)
Report Recall@5: 54.50% (109/200)
âœ… Saved: report_topk_optimized.csv


In [ ]:
print("\n" + "="*50)
print("PIPELINE COMPLETE")
print("="*50)
print(f"\nSchema Validation:")
print(f"  sensor_df columns: {sensor_df.columns.tolist()}")
print(f"  event_df columns: {event_df.columns.tolist()}")
print(f"\nArtifacts:")
print("  Saved: matches_contrastive.html (offline XY plot)")
print("  Saved: matches_topk.html (offline XY plot)")
print("  Saved: report_topk_results.csv")
print("  Saved: report_topk_enhanced.csv")
print("  Saved: report_topk_optimized.csv")
print(f"\nVisualization mode:")
print("  Offline only - no map tiles, no internet required")
print("  Shows tracking/report trajectories on XY axes")
print("  Self-contained Plotly HTML exports")


## Pipeline 5: Feature-Rich BiGRU Retrieval

Notebook nay duoc mo rong them mot pipeline retrieval moi dung `BiGRU + attention pooling` va feature chuoi giau ngu canh.

Muc tieu: thay the encoder MLP/feature tinh bang sequence encoder co kha nang hoc hinh dang quy dao va dong hoc tot hon.


In [ ]:
import pandas as pd

from gru_feature_pipeline import (
    build_gt_dict,
    evaluate_topk,
    export_topk_csv,
    run_pipeline_gru_feature_rich,
)

print("\n=== Pipeline 5: Feature-Rich BiGRU Retrieval ===")

if 'sensor_df' not in globals():
    sensor_df = pd.read_parquet('sensor.parquet')
if 'event_df' not in globals():
    event_df = pd.read_parquet('event.parquet')
if 'gt_df' not in globals():
    gt_df = pd.read_parquet('ground_truth.parquet')

gt_dict_gru = build_gt_dict(gt_df, event_df)
if not gt_dict_gru:
    raise ValueError('Could not build gt_dict from gt_df and event_df')

artifacts_gru = run_pipeline_gru_feature_rich(
    sensor_df,
    event_df,
    gt_dict_gru,
    top_k=5,
    min_len=6,
    max_len=18,
    stride=2,
    epochs=18,
    hidden_dim=96,
    emb_dim=128,
    batch_size=64,
    lr=3e-4,
)

report_to_tracks_gru = artifacts_gru['report_to_tracks']
recall_gru = evaluate_topk(report_to_tracks_gru, gt_dict_gru, k=5)
print(f"Feature-Rich BiGRU Recall@5: {recall_gru:.3f}")

export_topk_csv(report_to_tracks_gru, 'report_topk_gru_feature_rich.csv', top_k=5)
print('Saved: report_topk_gru_feature_rich.csv')


In [ ]:
print("\n=== GRU Pipeline Summary ===")
print(f"Triplets: {len(artifacts_gru['triplets'])}")
print(f"Sub-tracks: {len(artifacts_gru['sub_tracks'])}")
print(f"Feature dim: {artifacts_gru['sequence_bundle'].feature_dim}")
print(f"GRU Recall@5: {recall_gru:.2%}")


In [ ]:
print("\n=== Pipeline Comparison ===")
comparison_rows = []
if 'report_topk_results' in globals():
    comparison_rows.append(('Pipeline 2 - Triplet', evaluate_topk(report_topk_results, gt_dict, k=5)))
if 'report_topk_results_enhanced' in globals():
    comparison_rows.append(('Pipeline 3 - Enhanced', evaluate_topk(report_topk_results_enhanced, gt_dict, k=5)))
if 'report_topk_results_optimized' in globals():
    comparison_rows.append(('Pipeline 4 - Optimized', evaluate_topk(report_topk_results_optimized, gt_dict, k=5)))
if 'report_to_tracks_gru' in globals():
    comparison_rows.append(('Pipeline 5 - BiGRU', evaluate_topk(report_to_tracks_gru, gt_dict_gru, k=5)))

if comparison_rows:
    comparison_df = pd.DataFrame(comparison_rows, columns=["pipeline", "recall_at_5"]).sort_values("recall_at_5", ascending=False)
    display(comparison_df)
else:
    print("No pipeline outputs available for comparison.")
